# Llama-3.2 Korean Bllossom 3B — SFT for RC car commands

## 1. Install dependencies

In [ ]:
!python -m pip install bitsandbytes --prefer-binary --extra-index-url=https://jllllll.github.io/bitsandbytes-windows-webui

!pip install transformers accelerate
!pip install peft==0.10.0
!pip install -U trl
!pip install --upgrade datasets

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Paths & config

In [ ]:
import os, sys

DRIVE_ROOT       = '/content/drive/MyDrive'
BASE_DIR         = f'{DRIVE_ROOT}/capstone_design'
SRC_DIR          = f'{BASE_DIR}/llm_src'
BASE_MODEL       = 'Bllossom/llama-3.2-Korean-Bllossom-3B'
PAD_TOKEN        = '<|reserved_special_token_247|>'
TRAIN_JSONL      = f'{BASE_DIR}/data/total_commands.jsonl'
TEST_JSONL       = f'{BASE_DIR}/data/test.jsonl'
OUTPUT_DIR       = '/content/llm_fine_tune_output'
DRIVE_OUTPUT_DIR = f'{BASE_DIR}/outputs'
MERGED_DIR       = f'{BASE_DIR}/llama3.2-merged'
EVAL_CHECKPOINT  = OUTPUT_DIR

MAX_LENGTH  = 128
MAX_STEPS   = None
NUM_EPOCHS  = 1

assert os.path.isfile(f'{SRC_DIR}/llama3_2_sft.py'), f'Missing: {SRC_DIR}/llama3_2_sft.py'
assert os.path.isfile(TRAIN_JSONL), f'Missing: {TRAIN_JSONL}'

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Paths OK. Output:', OUTPUT_DIR)

## 4. Load model & tokenizer

In [ ]:
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=32,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    device_map={'': 0},
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.add_special_tokens({'pad_token': PAD_TOKEN})
model.config.pad_token_id = tokenizer.pad_token_id

print('Model ready on', model.device)

## 5. Dataset

In [ ]:
from llama3_2_sft import build_dataset

train_dataset, test_dataset = build_dataset(TRAIN_JSONL, tokenizer)

print(f'Train: {len(train_dataset)} | Test: {len(test_dataset)}')
print(train_dataset[0])

## 6. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir '{OUTPUT_DIR}/runs'

## 7. Train

In [ ]:
from llama3_2_sft import build_trainer

trainer = build_trainer(
    model, tokenizer, train_dataset, test_dataset, peft_config, OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    max_length=MAX_LENGTH,
)

train_stats = trainer.train()
trainer.save_model()
print('Adapter saved to', OUTPUT_DIR)

## 8. Copy checkpoints to Drive

In [ ]:
import os
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

!ls -lh {OUTPUT_DIR}
!cp -r {OUTPUT_DIR} {DRIVE_OUTPUT_DIR}

## 9. Merge LoRA into base (bf16) & save

추론용 병합 모델을 만든다. 4bit 베이스에는 LoRA를 병합할 수 없으므로 베이스를 bf16으로 다시 올린 뒤 병합한다.
결과 모델은 어댑터 연산 없이 바로 로드되며, 추론 시 4bit + PEFT 조합보다 빠르다.

In [ ]:
import gc, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

for name in ['model', 'base_model', 'trainer', 'llm_base']:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

merge_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map={'': 'cpu'},
)

merged_model = PeftModel.from_pretrained(
    merge_base, EVAL_CHECKPOINT, torch_dtype=torch.bfloat16
).merge_and_unload()
merged_model.eval()

merged_tokenizer = AutoTokenizer.from_pretrained(EVAL_CHECKPOINT)
merged_model.config.pad_token_id = merged_tokenizer.pad_token_id

os.makedirs(MERGED_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
merged_tokenizer.save_pretrained(MERGED_DIR)

print('Merged model saved to', MERGED_DIR)
!ls -lh {MERGED_DIR}